# Paper 2 · Langkah 8: latihan ulang dengan seed berbeda

**Pertanyaan:** apakah peta dan pola elemen desain berubah karena riwayat kompresi, atau cukup karena model dilatih ulang?

**Cara menjawab:** kondisi simetris dan kondisi satu sisi dilatih ulang dengan seed kedua, data dan fold-nya sama. Lalu dibandingkan:

1. **Kesepakatan peta** untuk gambar yang sama: kondisi sama dengan seed beda, lawan kondisi beda. Kalau keduanya mirip, ketidaksepakatan peta berasal dari latihan ulang saja.
2. **Pola elemen desain** pada karya manusia: apakah pola "detail" di kondisi satu sisi muncul lagi di seed kedua, dan apakah kondisi simetris tetap tanpa pola itu.

**Aturan baca yang dikunci sebelum hasil** (empat ukuran detail: gradasi, kepadatan tepi, tekstur kontras, homogenitas):
- Pola dianggap berasal dari kompresi kalau di seed kedua kondisi satu sisi mengulang arah yang sama dengan |r| ≥ 0,15 pada minimal 3 dari 4 ukuran, **dan** kondisi simetris punya |r| ≤ 0,10 pada minimal 3 dari 4 ukuran.
- Kalau salah satu syarat gagal, klaim di naskah harus dilemahkan.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**. Perkiraan sekitar 80 menit. Kondisi simetris dikerjakan lebih dulu. Hasil tiap bagian langsung disimpan ke Drive; kalau sesi terputus, klik Run all lagi.

## 1. Persiapan dan cek GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, random, shutil, zipfile, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.transforms.functional import gaussian_blur
import cv2, scipy, sklearn
from scipy import stats, ndimage
from PIL import Image
import PIL

# Pengaturan (sama dengan langkah 3 sampai 6; jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
SEED2 = 1042          # seed kedua untuk latihan ulang
INPUT = 224
PATCH, STRIDE = 32, 16
TOP = 0.20
N_RANDOM = 10
BLUR_KERNEL, BLUR_SIGMA = 21, 7.0
HUE_BINS, GLCM_LEVELS = 36, 32
CANNY_LOW, CANNY_HIGH = 100, 200
CHROMA_MIN = 0.10
MAX_EPOCHS, PATIENCE, BATCH, LR, LR_MIN, PRETRAINED = 30, 5, 32, 1e-4, 1e-6, True

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
RUNS = RESULTS / 'detektor_runs'
MODELS = RESULTS / 'models'
MAPS3 = RESULTS / 'peta_simetris'
MAPS2 = RESULTS / 'peta_logit'
MAPS4 = RESULTS / 'peta_seed2'
ELEM4 = RESULTS / 'elemen_seed2'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [RUNS, MODELS, MAPS4, ELEM4, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'
USE_AMP = device == 'cuda'

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}\nsklearn: {sklearn.__version__}\n'
       f'opencv: {cv2.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'latihan ulang dengan seed kedua: {SEED2} + nomor fold\n')
(NOTES / 'lingkungan_langkah8.txt').write_text(env)
print(env)

## 2. Data: versi simetris dan satu sisi

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)
assert (RESULTS / 'rq2_elemen_logit_per_gambar.csv').exists(), 'Jalankan notebook langkah 6 dulu.'
assert (RESULTS / 'rq2_elemen_simetris_per_gambar.csv').exists(), 'Jalankan notebook langkah 7 dulu.'

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)

def enc(im, q, sub):
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q, subsampling=sub, optimize=False); return buf.getvalue()
def dec(b):
    im = Image.open(io.BytesIO(b)); im.load(); return im.convert('RGB')
def square256(im):
    im = im.convert('RGB'); w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    return im

def make_version(row, version):
    with Image.open(row.raw_path) as im:
        im.load(); im = im.convert('RGB')
    if version == 'riwayat_disamakan' and row.source == 'human':
        im = dec(enc(im, 75, 2))
    im = square256(im)
    if version == 'simetris':
        im = dec(enc(im, 75, 2))
    b = enc(im, 90, 0)
    out = dec(b)
    if out.size != (INPUT, INPUT):
        out = out.resize((INPUT, INPUT), Image.BICUBIC)
    return np.asarray(out, dtype=np.uint8), len(b)

VERSIONS = ['simetris', 'riwayat_disamakan']
XH, XT, SIZE = {}, {}, {}
for v in VERSIONS:
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(lambda r: make_version(r, v), manifest.itertuples()))
    XH[v] = np.stack([a for a, _ in res]); SIZE[v] = np.array([s for _, s in res]) / 1024
    XT[v] = torch.from_numpy(XH[v]).permute(0, 3, 1, 2).contiguous()
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
src_all, folds, y_all = manifest['source'].values, manifest['fold'].values, manifest['y'].values
strata_all = (manifest['source'] + '|' + manifest['style']).values
print(f'Data siap ({time.time() - t0:.0f} detik):', {v: XH[v].shape for v in VERSIONS})

## 3. Fungsi model, latihan, dan peta (sama dengan langkah 7)

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
NPIX = INPUT * INPUT
FRACS = torch.arange(1, 11, device=device) * 0.05

def norm_t(xb):
    return ((xb.to(device).float().div(255) - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def build_model(pretrained=False):
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device).to(memory_format=torch.channels_last)

def load_model(name):
    m = build_model(False)
    m.load_state_dict(torch.load(MODELS / f'{name}.pt', map_location=device))
    return m.eval()

@torch.no_grad()
def probs(model, xn, bs=256):
    out = []
    for i in range(0, len(xn), bs):
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(xn[i:i + bs])
        out.append(torch.softmax(logits.float(), 1))
    return torch.cat(out)

@torch.no_grad()
def predict_ai(model, X, idx):
    return torch.cat([probs(model, norm_t(X[idx[i:i + 128]])) for i in range(0, len(idx), 128)]).cpu().numpy()

@torch.no_grad()
def logits32(model, xn, bs=256):
    return torch.cat([model(xn[i:i + bs]).float() for i in range(0, len(xn), bs)])

# --- latihan (sama dengan langkah 3 dan 4) ---
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def prep_train(xb):
    xb = xb.to(device, non_blocking=True).float().div_(255)
    flip = torch.rand(xb.size(0), device=device) < 0.5
    xb[flip] = xb[flip].flip(-1)
    return ((xb - MEAN) / STD).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def val_loss(model, X, idx):
    model.eval(); tot = 0.0
    for i in range(0, len(idx), 128):
        b = idx[i:i + 128]
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(norm_t(X[b]))
        tot += F.cross_entropy(logits.float(), torch.as_tensor(y_all[b], device=device), reduction='sum').item()
    return tot / len(idx)

def train_model(X, tr_idx, va_idx, seed):
    set_seed(seed)
    model = build_model(PRETRAINED)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    rng = np.random.default_rng(seed)
    best, best_state, bad, hist = float('inf'), None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); perm = rng.permutation(tr_idx)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=USE_AMP):
                loss = F.cross_entropy(model(prep_train(X[b])), torch.as_tensor(y_all[b], device=device))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v = val_loss(model, X, va_idx); hist.append(round(v, 4))
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model.eval(), hist

def inner_split(tr_idx, seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    try:
        a, b = next(sss.split(tr_idx, strata_all[tr_idx]))
    except ValueError:
        a, b = next(sss.split(tr_idx, src_all[tr_idx]))
    return tr_idx[a], tr_idx[b]

def source_rates(p_ai, idx):
    pred = (p_ai >= 0.5).astype(int); s = src_all[idx]; y = y_all[idx]; out = {'akurasi': float((pred == y).mean())}
    if len(np.unique(y)) == 2: out['auc'] = float(roc_auc_score(y, p_ai))
    for k, name in (('human', 'benar_manusia'), ('latent_diffusion', 'benar_LD'), ('stable_diffusion', 'benar_SD')):
        if (s == k).any(): out[name] = float((pred[s == k] == y[s == k]).mean())
    return out

# --- peta occlusion skala logit dan uji deletion (sama dengan langkah 6) ---
pos = list(range(0, INPUT - PATCH + 1, STRIDE)); N1 = len(pos)
M1 = torch.zeros(N1 * N1, 1, INPUT, INPUT, dtype=torch.bool, device=device); k = 0
for i in pos:
    for j in pos:
        M1[k, 0, i:i + PATCH, j:j + PATCH] = True; k += 1

@torch.no_grad()
def occlusion_logit(model, xn, pred):
    out, m0s = [], []
    for i in range(len(xn)):
        x = xn[i:i + 1]; c = int(pred[i]); mu = x.mean((2, 3), keepdim=True)
        z0 = logits32(model, x)[0]; m0 = z0[c] - z0[1 - c]
        z = logits32(model, torch.where(M1, mu, x))
        out.append((m0 - (z[:, c] - z[:, 1 - c])).reshape(N1, N1)); m0s.append(m0)
    return torch.stack(out), torch.stack(m0s)

def patch_to_cells(d):
    kk, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C, device=d.device); cnt = torch.zeros(C, C, device=d.device)
    for a in range(kk):
        for b in range(kk):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_desc(maps, gen):
    flat = maps.reshape(len(maps), -1)
    flat = flat + 1e-7 * flat.abs().amax(1, keepdim=True).clamp(min=1e-12) * torch.rand(flat.shape, device=flat.device, generator=gen)
    order = flat.argsort(1, descending=True)
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(order.shape[1], device=flat.device).expand_as(order))
    return rank

yy, xx = torch.meshgrid(torch.arange(INPUT, device=device), torch.arange(INPUT, device=device), indexing='ij')
c0 = (INPUT - 1) / 2
CENTRE = 1 - torch.sqrt((yy - c0) ** 2 + (xx - c0) ** 2) / (c0 * 2 ** 0.5)

@torch.no_grad()
def deletion(model, x, maps, c, p0, gen):
    rank = rank_desc(maps, gen)
    thr = (FRACS * NPIX).long()
    mask = (rank[:, None, :] < thr[None, :, None]).reshape(-1, 1, INPUT, INPUT)
    out = {}
    for name, rep in (('blur', gaussian_blur(x, [BLUR_KERNEL, BLUR_KERNEL], [BLUR_SIGMA, BLUR_SIGMA])), ('mean', x.mean((2, 3), keepdim=True))):
        p = probs(model, torch.where(mask, rep, x))[:, c].reshape(len(maps), -1)
        out[name] = (0.5 * p0 + p[:, :-1].sum(1) + 0.5 * p[:, -1]) / len(FRACS)
    return out
print('Fungsi siap. Posisi kotak occlusion:', N1, 'x', N1)

## 4. Latih ulang 10 model dengan seed kedua

In [ ]:
SRCMAP = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
UNITS = [(cond, k) for cond in VERSIONS for k in range(5)]        # simetris dulu, lalu satu sisi
def unit_name(cond, k): return f'{cond}_s2_fold{k}'

for cond, k in UNITS:
    name = unit_name(cond, k)
    if (MODELS / f'{name}.pt').exists() and (RUNS / f'{name}.json').exists():
        print(f'{name}: sudah selesai, dilewati.'); continue
    t0 = time.time(); X = XT[cond]; seed = SEED2 + k
    tr_idx, te_idx = np.where(folds != k)[0], np.where(folds == k)[0]
    tr, va = inner_split(tr_idx, seed)
    model, hist = train_model(X, tr, va, seed)
    p = predict_ai(model, X, te_idx)[:, 1]
    rec = {'nama': name, 'kondisi': cond, 'seed': seed, 'epoch': len(hist), 'detik': round(time.time() - t0), **source_rates(p, te_idx)}
    torch.save(model.state_dict(), MODELS / f'{name}.pt'); (RUNS / f'{name}.json').write_text(json.dumps(rec))
    print(f"{name}: akurasi {rec['akurasi']:.3f}, AUC {rec['auc']:.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
    del model; torch.cuda.empty_cache()

## 5. Peta occlusion skala logit (bagian terlama)

In [ ]:
def process_maps(cond, k):
    name = unit_name(cond, k); f = MAPS4 / f'{name}.npz'
    if f.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time(); model = load_model(name); idx = np.where(folds == k)[0]; X = XT[cond]
    P, OCC, M0 = [], [], []
    for s in range(0, len(idx), 16):
        b = idx[s:s + 16]; xn = norm_t(X[b]); pr = probs(model, xn); pred = pr.argmax(1)
        d, m0 = occlusion_logit(model, xn, pred.tolist())
        P.append(pr.cpu()); OCC.append(d.cpu()); M0.append(m0.cpu())
    P = torch.cat(P).numpy()
    np.savez_compressed(f, sample_id=manifest['sample_id'].values[idx].astype(str), p=P, pred=P.argmax(1),
                        margin0=torch.cat(M0).numpy().astype(np.float32), occ_logit=torch.cat(OCC).numpy().astype(np.float32))
    print(f'{name}: {len(idx)} gambar, {time.time() - t0:.0f} detik')
    del model; torch.cuda.empty_cache()

for cond, k in UNITS:
    process_maps(cond, k)

## 6. Elemen desain

In [ ]:
NPIX = INPUT * INPUT
MEASURES = ['hue_entropy', 'saturation', 'colourfulness', 'brightness', 'contrast', 'gradient',
            'edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
LABEL = {'hue_entropy': 'Hue: entropi', 'saturation': 'Warna: saturasi', 'colourfulness': 'Warna: kekayaan',
         'brightness': 'Nada: kecerahan', 'contrast': 'Nada: kontras', 'gradient': 'Nada: gradasi',
         'edge_density': 'Garis: kepadatan tepi', 'glcm_contrast': 'Tekstur: kontras',
         'glcm_homogeneity': 'Tekstur: homogenitas', 'glcm_energy': 'Tekstur: energi', 'glcm_correlation': 'Tekstur: korelasi'}

# Pasangan piksel bertetangga untuk GLCM (4 arah, jarak 1)
_flat = np.arange(NPIX).reshape(INPUT, INPUT)
PAIRS = []
for dy, dx in [(0, 1), (1, 0), (1, 1), (1, -1)]:
    a = _flat[max(0, -dy):INPUT - max(0, dy), max(0, -dx):INPUT - max(0, dx)]
    b = _flat[max(0, dy):INPUT - max(0, -dy), max(0, dx):INPUT - max(0, -dx)]
    PAIRS.append((a.ravel(), b.ravel()))
_lv = np.arange(GLCM_LEVELS, dtype=np.float64)
_I, _J = np.meshgrid(_lv, _lv, indexing='ij')
_D2 = (_I - _J) ** 2

def precompute(img):
    f = {}
    rgb = img.astype(np.float32)
    hsv = cv2.cvtColor(rgb / 255.0, cv2.COLOR_RGB2HSV)            # H 0-360, S 0-1, V 0-1
    f['hbin'] = np.minimum((hsv[..., 0] / 360.0 * HUE_BINS).astype(np.int64), HUE_BINS - 1).ravel()
    f['sat'] = hsv[..., 1].ravel(); f['val'] = hsv[..., 2].ravel()
    f['chroma_ok'] = (f['sat'] > CHROMA_MIN) & (f['val'] > CHROMA_MIN)
    f['rg'] = (rgb[..., 0] - rgb[..., 1]).ravel()
    f['yb'] = (0.5 * (rgb[..., 0] + rgb[..., 1]) - rgb[..., 2]).ravel()
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f['lum'] = gray.astype(np.float32).ravel()
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    f['grad'] = np.sqrt(gx ** 2 + gy ** 2).ravel()
    f['edge'] = (cv2.Canny(gray, CANNY_LOW, CANNY_HIGH) > 0).ravel()
    q = (gray.astype(np.int64) * GLCM_LEVELS // 256).ravel()
    f['codes'] = [q[a] * GLCM_LEVELS + q[b] for a, b in PAIRS]
    return f

def region_measures(f, m):
    out = dict.fromkeys(MEASURES, np.nan)
    if m.sum() < 100:
        return out
    sel = m & f['chroma_ok']
    if sel.sum() >= 50:
        h = np.bincount(f['hbin'][sel], minlength=HUE_BINS).astype(np.float64); p = h / h.sum(); p = p[p > 0]
        out['hue_entropy'] = float(-(p * np.log2(p)).sum())
    out['saturation'] = float(f['sat'][m].mean())
    rg, yb = f['rg'][m], f['yb'][m]
    out['colourfulness'] = float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))
    out['brightness'] = float(f['val'][m].mean())
    out['contrast'] = float(f['lum'][m].std())
    out['gradient'] = float(f['grad'][m].mean())
    out['edge_density'] = float(f['edge'][m].mean())
    P = np.zeros(GLCM_LEVELS * GLCM_LEVELS)
    for (a, b), code in zip(PAIRS, f['codes']):
        ok = m[a] & m[b]
        P += np.bincount(code[ok], minlength=GLCM_LEVELS * GLCM_LEVELS)
    if P.sum() > 0:
        P = P.reshape(GLCM_LEVELS, GLCM_LEVELS); P = P + P.T; P /= P.sum()
        out['glcm_contrast'] = float((P * _D2).sum())
        out['glcm_homogeneity'] = float((P / (1 + _D2)).sum())
        out['glcm_energy'] = float(np.sqrt((P ** 2).sum()))
        mu = (P.sum(1) * _lv).sum(); var = (P.sum(1) * (_lv - mu) ** 2).sum()
        out['glcm_correlation'] = float((P * (_I - mu) * (_J - mu)).sum() / var) if var > 1e-12 else np.nan
    return out

GRID = max(1, INPUT // 32)
def composition(m):
    m2 = m.reshape(INPUT, INPUT); ys, xs = np.nonzero(m2); c = (INPUT - 1) / 2
    cell = m2[:GRID * (INPUT // GRID), :GRID * (INPUT // GRID)].reshape(GRID, INPUT // GRID, GRID, INPUT // GRID).sum((1, 3)).ravel().astype(np.float64)
    p = cell / cell.sum(); p = p[p > 0]
    lab, ncomp = ndimage.label(m2, structure=np.ones((3, 3)))
    sizes = np.bincount(lab.ravel())[1:]
    return {'comp_centroid_dist': float(np.sqrt((ys.mean() - c) ** 2 + (xs.mean() - c) ** 2) / (c * 2 ** 0.5)),
            'comp_spatial_entropy': float(-(p * np.log2(p)).sum() / np.log2(GRID * GRID)) if GRID > 1 else np.nan,
            'comp_n_components': int(ncomp), 'comp_largest_share': float(sizes.max() / sizes.sum())}

def patch_to_cells_cpu(d):
    k, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C); cnt = torch.zeros(C, C)
    for a in range(k):
        for b in range(k):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_maps(maps, rng):
    n = len(maps); rank = np.empty((n, NPIX), dtype=np.uint16)
    for s in range(0, n, 200):
        flat = maps[s:s + 200].reshape(-1, NPIX).astype(np.float64)
        flat = flat + 1e-9 * rng.random(flat.shape)                 # pemecah nilai kembar
        order = np.argsort(-flat, axis=1)
        np.put_along_axis(rank[s:s + 200], order, np.arange(NPIX, dtype=np.uint16)[None].repeat(len(order), 0), axis=1)
    return rank
ALL = np.ones(NPIX, dtype=bool)


def process_elem(cond, k):
    name = unit_name(cond, k); out_file = ELEM4 / f'{name}.csv'
    if out_file.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    if not (MAPS4 / f'{name}.npz').exists():
        print(f'{name}: peta belum ada, dilewati.'); return
    t0 = time.time()
    z = np.load(MAPS4 / f'{name}.npz', allow_pickle=True)
    ids = z['sample_id'].astype(str); idx = np.array([idx_of[s] for s in ids]); pred = z['pred'].astype(int); n = len(idx)
    rng = np.random.default_rng(SEED + sum(map(ord, name)))
    occ = torch.from_numpy(z['occ_logit']).float(); cells = patch_to_cells_cpu(occ).clamp(min=0)
    empty = (cells.amax((1, 2)) <= 0).numpy(); ranks = rank_maps(up(cells).numpy(), rng)
    partner = np.arange(n); s_here = src_all[idx]
    for s in np.unique(s_here):
        loc = rng.permutation(np.where(s_here == s)[0])
        if len(loc) > 1: partner[loc] = np.roll(loc, -1)
    rows = []
    for j in range(n):
        f = precompute(XH[cond][idx[j]])
        row = {'bagian': name, 'kondisi': cond, 'sample_id': ids[j], 'pred': int(pred[j]), 'peta_kosong': bool(empty[j])}
        if not empty[j]:
            m = ranks[j] < TOP * NPIX
            for tag, mask in (('in', m), ('out', ~m)):
                row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
            if partner[j] != j and not empty[partner[j]]:
                ms = ranks[partner[j]] < TOP * NPIX
                for tag, mask in (('shin', ms), ('shout', ~ms)):
                    row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
        rows.append(row)
    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')

for cond, k in UNITS:
    process_elem(cond, k)

## 7. Ringkasan

- **T1**: kinerja model seed kedua.
- **T2**: kesepakatan peta, kondisi sama dengan seed beda lawan kondisi beda.
- **T3**: elemen desain untuk empat set (simetris A dan B, satu sisi A dan B).
- **T4**: pemeriksaan aturan baca.

In [ ]:
NAMA = {'simetris': 'simetris', 'riwayat_disamakan': 'satu sisi'}
GROUPS = ['manusia benar', 'LD benar', 'SD benar']
DETAIL = ['gradient', 'edge_density', 'glcm_contrast', 'glcm_homogeneity']

# ---- T1. kinerja seed kedua ----
rows = []
for cond in VERSIONS:
    r2 = [json.loads(f.read_text()) for f in sorted(RUNS.glob(f'{cond}_s2_fold*.json'))]
    if r2:
        d = pd.DataFrame(r2)[['akurasi', 'auc', 'benar_manusia', 'benar_LD', 'benar_SD']].mean()
        rows.append({'kondisi': NAMA[cond], 'seed': 'kedua', 'model': len(r2), **d.to_dict()})
print('T1. KINERJA MODEL SEED KEDUA (rata-rata fold). Pembanding seed pertama: simetris akurasi 0,976; satu sisi 0,980')
print(pd.DataFrame(rows).round(3).to_string(index=False))

# ---- muat peta ----
def load_maps(folder, pattern):
    ids, preds, cells = [], [], []
    for k in range(5):
        f = folder / pattern.format(k=k)
        if not f.exists(): return None
        z = np.load(f, allow_pickle=True)
        ids.append(z['sample_id'].astype(str)); preds.append(z['pred'].astype(int))
        cells.append(patch_to_cells_cpu(torch.from_numpy(z['occ_logit']).float()).clamp(min=0))
    ids = np.concatenate(ids); o = np.argsort(ids)
    return ids[o], np.concatenate(preds)[o], torch.cat(cells)[o]
MAPSETS = {'simetris A': load_maps(MAPS3, 'simetris_fold{k}.npz'), 'simetris B': load_maps(MAPS4, 'simetris_s2_fold{k}.npz'),
           'satu sisi A': load_maps(MAPS2, 'riwayat_disamakan_fold{k}.npz'), 'satu sisi B': load_maps(MAPS4, 'riwayat_disamakan_s2_fold{k}.npz')}
MAPSETS = {k: v for k, v in MAPSETS.items() if v is not None}
base_ids = MAPSETS['simetris A'][0]
info = manifest.set_index('sample_id').loc[base_ids]; yv, sv = info['y'].values, info['source'].values
gen = torch.Generator().manual_seed(SEED)
def region(cells):
    u = up(cells).reshape(len(cells), -1)
    u = u + 1e-7 * u.abs().amax(1, keepdim=True).clamp(min=1e-12) * torch.rand(u.shape, generator=gen)
    thr = u.kthvalue(int(round((1 - TOP) * u.shape[1])), dim=1, keepdim=True).values
    return u > thr
OK, REG = {}, {}
for name, (ids, pred, cells) in MAPSETS.items():
    assert (ids == base_ids).all()
    OK[name] = (pred == yv) & (cells.amax((1, 2)).numpy() > 0); REG[name] = region(cells)

PAIRS = [('simetris A', 'simetris B', 'kondisi sama, seed beda'), ('satu sisi A', 'satu sisi B', 'kondisi sama, seed beda'),
         ('satu sisi A', 'simetris A', 'kondisi beda'), ('satu sisi B', 'simetris B', 'kondisi beda'),
         ('satu sisi A', 'simetris B', 'kondisi beda'), ('satu sisi B', 'simetris A', 'kondisi beda')]
rows = []
for a, b, kind in PAIRS:
    if a not in MAPSETS or b not in MAPSETS: continue
    both = OK[a] & OK[b]; idx = np.where(both)[0]
    rho = np.full(len(base_ids), np.nan); iou = np.full(len(base_ids), np.nan)
    if len(idx):
        A_, B_ = MAPSETS[a][2][idx].reshape(len(idx), -1).numpy(), MAPSETS[b][2][idx].reshape(len(idx), -1).numpy()
        rho[idx] = [stats.spearmanr(A_[i], B_[i]).statistic for i in range(len(idx))]
        iou[idx] = ((REG[a][idx] & REG[b][idx]).sum(1).float() / (REG[a][idx] | REG[b][idx]).sum(1).float()).numpy()
    for s in SRCMAP:
        sel = both & (sv == s)
        rows.append({'jenis': kind, 'pasangan': f'{a} vs {b}', 'sumber': SRCMAP[s], 'n': int(sel.sum()),
                     'median_korelasi': np.nanmedian(rho[sel]) if sel.any() else np.nan, 'median_iou': np.nanmedian(iou[sel]) if sel.any() else np.nan})
T2 = pd.DataFrame(rows); T2.to_csv(RESULTS / 'rq2_seed_kesepakatan_peta.csv', index=False)
print('\nT2. KESEPAKATAN PETA UNTUK GAMBAR YANG SAMA (median korelasi Spearman | median IoU area 20%; IoU acak sekitar 0,11)')
t = T2.copy(); t['sel'] = t.median_korelasi.map('{:.3f}'.format) + ' | ' + t.median_iou.map('{:.3f}'.format)
print(t.pivot(index=['jenis', 'pasangan'], columns='sumber', values='sel').reindex(columns=['manusia', 'LD', 'SD']).to_string())

# ---- T3. elemen desain: seed A dan seed B ----
def paired_effect(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; x = x[x != 0]
    if len(x) < 10: return np.nan, np.nan, len(x)
    r = stats.rankdata(np.abs(x)); pos_, neg_ = r[x > 0].sum(), r[x < 0].sum()
    return (pos_ - neg_) / (pos_ + neg_), stats.wilcoxon(x).pvalue, len(x)
def bh(p):
    p = np.asarray(p, dtype=float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    if ok.sum() == 0: return out
    pv = p[ok]; order = np.argsort(pv); ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]; res = np.empty(len(pv)); res[order] = np.minimum(ranked, 1)
    out[ok] = res; return out

def effects(d, label):
    d = d.drop(columns=['source', 'y'], errors='ignore').join(manifest.set_index('sample_id')[['source', 'y']], on='sample_id')
    d = d[(d.pred == d.y) & ~d.peta_kosong]
    rows = []
    for s, g in d.groupby('source'):
        for m in MEASURES:
            r, p, n = paired_effect((g[f'in_{m}'] - g[f'out_{m}']) - (g[f'shin_{m}'] - g[f'shout_{m}']))
            rows.append({'set': label, 'kelompok': SRCMAP[s] + ' benar', 'ukuran': m, 'n': n, 'r': r, 'p': p})
    out = pd.DataFrame(rows)
    if len(out): out['bermakna'] = bh(out.p.values) < 0.05
    return out

sets = [effects(pd.read_csv(RESULTS / 'rq2_elemen_simetris_per_gambar.csv'), 'simetris A')]
o1 = pd.read_csv(RESULTS / 'rq2_elemen_logit_per_gambar.csv'); o1 = o1[o1.bagian.str.startswith('riwayat_disamakan_fold')]
sets.append(effects(o1, 'satu sisi A'))
for cond in VERSIONS:
    fs = sorted(ELEM4.glob(f'{cond}_s2_fold*.csv'))
    if len(fs) == 5: sets.append(effects(pd.concat([pd.read_csv(f) for f in fs], ignore_index=True), NAMA[cond] + ' B'))
    else: print(f'Elemen seed kedua untuk kondisi {NAMA[cond]} belum lengkap ({len(fs)} dari 5 fold).')
E = pd.concat([x for x in sets if len(x)], ignore_index=True); E.to_csv(RESULTS / 'rq2_seed_elemen.csv', index=False)
E['sel'] = [('nan' if pd.isna(r) else f'{r:+.2f}') + ('*' if b else ' ') for r, b in zip(E.r, E.bermakna)]
ORDER = [s for s in ['simetris A', 'simetris B', 'satu sisi A', 'satu sisi B'] if s in E.set.unique()]
for grp in GROUPS:
    if not (E.kelompok == grp).any(): continue
    pv = E[E.kelompok == grp].pivot(index='ukuran', columns='set', values='sel').reindex(MEASURES).reindex(columns=ORDER)
    pv.index = [LABEL[i] for i in pv.index]
    print(f'\nT3. ELEMEN DESAIN, {grp.upper()} (r; * = bermakna FDR 5% dalam tiap set; A = seed pertama, B = seed kedua)')
    print(pv.to_string())

# ---- T4. aturan baca yang dikunci sebelum hasil ----
h = E[(E.kelompok == 'manusia benar') & E.ukuran.isin(DETAIL)].pivot(index='ukuran', columns='set', values='r').reindex(DETAIL)
print('\nT4. ATURAN BACA untuk empat ukuran detail pada karya manusia (gradasi, tepi, tekstur kontras, homogenitas)')
print(h.round(3).to_string())
if 'simetris B' in h:
    print(f"   Simetris B: {int((h['simetris B'].abs() <= 0.10).sum())} dari 4 ukuran punya |r| <= 0,10 (syarat: minimal 3)")
if 'satu sisi B' in h and 'satu sisi A' in h:
    same = (np.sign(h['satu sisi B']) == np.sign(h['satu sisi A'])) & (h['satu sisi B'].abs() >= 0.15)
    print(f"   Satu sisi B: {int(same.sum())} dari 4 ukuran searah dengan seed pertama dan |r| >= 0,15 (syarat: minimal 3)")
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tabel T1 sampai T4 muncul, semua beres. Tekan **Cmd+S**, lalu kirim link Colab-nya ke Claude. Kalau GPU habis di tengah jalan, kirim saja link-nya apa adanya: hasil kondisi simetris sudah cukup untuk dibaca lebih dulu.